In [6]:
import pandas as pd
import numpy as np

cols = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
        "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
df = pd.read_csv("../data/processed.cleveland.data",
                 header=None, names=cols, na_values="?")
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)          # 应为 (242, 13) 与 (61, 13)
print(y_train.mean(), y_test.mean())        # 两集中有病比例都应 ≈ 0.459

(242, 13) (61, 13)
0.45867768595041325 0.45901639344262296


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

scale_cols  = ["age", "trestbps", "chol", "thalach", "oldpeak"]  # 数值：填补+标准化
onehot_cols = ["cp", "restecg", "thal"]                          # 无序类别：填补+独热
pass_cols   = ["sex", "fbs", "exang", "slope", "ca"]             # 二值/有序：原样通过

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), scale_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(drop="first"))]), onehot_cols),
    ("pass", SimpleImputer(strategy="most_frequent"), pass_cols),
])

In [11]:
from sklearn.linear_model import LogisticRegression

pipe_test = Pipeline([("prep", preprocess),
                      ("clf", LogisticRegression(max_iter=1000, random_state=42))])
pipe_test.fit(X_train, y_train)
print("快速验证，测试集准确率:", pipe_test.score(X_test, y_test))

快速验证，测试集准确率: 0.8852459016393442
